# Notebook 01 — Attention by Hand

**Prerequisite:** you have read `theory/01_scaled_dot_product.md` and attempted exercises 1.1–1.5.

## Question
What does $Q K^\top$ actually *measure*, and how does softmax turn it into a lookup?

## Hypothesis (write yours before running anything)
_Your prediction here:_
1. If a query exactly matches one key, the attention distribution will ...
2. If we scale Q by λ → ∞, the output will ...
3. If all keys are identical, the output will ...

## Method
Implement scaled dot-product attention in pure NumPy, then PyTorch. Visualize the attention matrix on toy inputs, then perturb.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import matplotlib.pyplot as plt

from attn_lab import plot_attention, plot_softmax_bars, get_device

device = get_device()
print('device:', device)
np.set_printoptions(precision=3, suppress=True)

## 1. Scaled dot-product attention in NumPy

Implement it from the definition — don't import any attention utility. This is the single most important function in the transformer; write it yourself.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)  # shift invariance (Prop 2.1)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)        # (n, m)
    weights = softmax(scores, axis=-1)     # (n, m)
    output = weights @ V                   # (n, d_v)
    return output, weights

## 2. Verify the worked example from §1.4

You should get outputs $\approx (5, 5)$ and $\approx (3.30, 6.70)$.

In [ ]:
Q = np.array([[1., 0.], [0., 1.]])
K = np.array([[1., 0.], [1., 1.]])
V = np.array([[10., 0.], [0., 10.]])

out, w = attention(Q, K, V)
print('weights:\n', w)
print('output:\n', out)

## 3. Perfect-match retrieval

Set up 4 orthogonal keys and a query equal to key 2. What should the attention distribution look like?

In [ ]:
d_k = 8
K = np.eye(4, d_k)           # 4 orthogonal keys (one-hot in R^d_k)
V = np.array([[1., 0., 0.],
              [0., 1., 0.],
              [0., 0., 1.],
              [9., 9., 9.]])

q = K[2:3].copy()            # query = key 2
out, w = attention(q, K, V)

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
plot_softmax_bars(w[0], title='attention weights for q = k_2', ax=axes[0])
plot_attention(w, title='(1 query x 4 keys)', ax=axes[1])
plt.tight_layout(); plt.show()
print('output:', out[0])

**Try it yourself:** what happens if you scale `q` by 10? By 100? What does this tell you about the relationship between **query magnitude** and **attention sharpness**?

In [ ]:
for scale in [0.1, 1.0, 10.0, 100.0]:
    _, w = attention(scale * q, K, V)
    entropy = -(w * np.log(w + 1e-12)).sum()
    print(f'scale={scale:>6} | top weight={w[0].max():.4f} | entropy={entropy:.4f}')

## 4. Self-attention on a toy sentence

Build a 5-token sequence where token embeddings are hand-crafted so that token 0 is similar to token 3, and token 1 is similar to token 4. Verify that self-attention recovers this structure as an off-diagonal pattern in the attention matrix.

In [ ]:
rng = np.random.default_rng(0)
d = 16

# Handcrafted embeddings: tokens 0 & 3 share a signal; 1 & 4 share another; 2 is alone.
e0 = rng.normal(size=d)
e1 = rng.normal(size=d)
e2 = rng.normal(size=d)
X = np.stack([e0, e1, e2, e0 + 0.1*rng.normal(size=d), e1 + 0.1*rng.normal(size=d)])

# For this demo we skip learned projections — Q=K=V=X.
_, W = attention(X, X, X)
plot_attention(W, xlabels=['t0','t1','t2','t3','t4'], ylabels=['t0','t1','t2','t3','t4'],
               title='self-attention weights (Q=K=V=X)')
plt.show()

**Your turn (Exercise N.1):** Modify `X` so that token 2 is similar to *both* 0 and 4. Predict what the attention row for token 2 will look like, then verify.

**Your turn (Exercise N.2):** Add learned projections `W_Q, W_K, W_V` (random for now). How does the pattern change? What changes if you tie `W_Q = W_K`?

## 5. PyTorch version + a gradient check

Rewrite in torch and verify autograd gives the right Jacobian for softmax (Prop 2.2).

In [ ]:
import torch.nn.functional as F

def attention_t(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-1, -2) / d_k**0.5
    w = scores.softmax(dim=-1)
    return w @ V, w

torch.manual_seed(0)
Qt = torch.randn(2, 4, requires_grad=False)
Kt = torch.randn(3, 4)
Vt = torch.randn(3, 5)
out_t, w_t = attention_t(Qt, Kt, Vt)
out_np, w_np = attention(Qt.numpy(), Kt.numpy(), Vt.numpy())
print('numpy/torch max abs diff:', np.max(np.abs(out_t.numpy() - out_np)))

In [ ]:
# Verify Prop 2.2: dp_i / ds_j = p_i (delta_ij - p_j)
s = torch.randn(5, requires_grad=True)
p = s.softmax(dim=-1)
J_autograd = torch.autograd.functional.jacobian(lambda x: x.softmax(-1), s)
J_formula = torch.diag(p) - p.unsqueeze(1) * p.unsqueeze(0)
print('max abs diff:', (J_autograd - J_formula).abs().max().item())

## Finding

_2–3 sentences on what surprised you, what you got wrong in your hypothesis, and what felt obvious after doing it. Copy to `../learnings.md`._